# Labeling agreement

This notebook uses the original `ecb_random_tolabel.csv` and `fed_random_tolabel.csv` files as the base sample, then left-joins Daniel, Eric, and Sam's labels onto those rows by `sentence_id`. The goal is to keep the original 100 ECB and 100 Fed sentences as the reference set and inspect any missing or duplicate labels directly.

In [19]:
from itertools import combinations
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../data/csv")
BASE_FILES = {
    "ecb": DATA_DIR / "ecb_random_tolabel.csv",
    "fed": DATA_DIR / "fed_random_tolabel.csv",
}
LABEL_FILES = {
    "daniel": {
        "ecb": DATA_DIR / "ecb_random_tolabel_daniel.csv",
        "fed": DATA_DIR / "fed_random_tolabel_daniel.csv",
    },
    "eric": {
        "ecb": DATA_DIR / "ecb_random_tolabel_eric.csv",
        "fed": DATA_DIR / "fed_random_tolabel_eric.csv",
    },
    "sam": {
        "ecb": DATA_DIR / "ecb_random_tolabel_sam.csv",
        "fed": DATA_DIR / "fed_random_tolabel_sam.csv",
    },
}
LABELERS = ["daniel", "eric", "sam"]

pd.set_option("display.max_colwidth", 120)

In [20]:
def detect_separator(path: Path) -> str:
    header = path.open("r", encoding="utf-8", errors="replace").readline()
    return ";" if header.count(";") > header.count(",") else ","


def read_csv_clean(path: Path) -> pd.DataFrame:
    sep = detect_separator(path)
    df = pd.read_csv(path, sep=sep, encoding="utf-8", engine="python")
    df = df.loc[:, ~df.columns.str.startswith("Unnamed:")].copy()
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
    return df


def read_base_sample(bank: str) -> pd.DataFrame:
    df = read_csv_clean(BASE_FILES[bank]).copy()
    df["bank"] = bank
    return df


def read_labeler_labels(bank: str, labeler: str) -> pd.DataFrame:
    df = read_csv_clean(LABEL_FILES[labeler][bank]).copy()
    label_col = f"{labeler}_label"
    df[label_col] = df["label"].astype("string").str.strip().str.lower()
    return df[["sentence_id", label_col]]


def cohens_kappa(left: pd.Series, right: pd.Series) -> float:
    paired = pd.DataFrame({"left": left, "right": right}).dropna()
    if paired.empty:
        return float("nan")

    agreement = (paired["left"] == paired["right"]).mean()
    confusion = pd.crosstab(paired["left"], paired["right"])
    total = confusion.to_numpy().sum()
    row_probs = confusion.sum(axis=1) / total
    col_probs = confusion.sum(axis=0) / total
    expected = row_probs.mul(col_probs, fill_value=0).sum()
    if expected == 1:
        return float("nan")
    return (agreement - expected) / (1 - expected)

In [21]:
base_frames = [read_base_sample(bank) for bank in ["ecb", "fed"]]
base_sample = pd.concat(base_frames, ignore_index=True).sort_values(["bank", "sentence_id"]).reset_index(drop=True)
base_sample[["bank", "sentence_id", "label"]].head()

,bank,sentence_id,label
0,ecb,20150305-ecb_105,NaN
1,ecb,20150415-ecb_249,NaN
2,ecb,20150415-ecb_83,NaN
3,ecb,20150603-ecb_117,NaN
4,ecb,20150603-ecb_202,NaN


In [22]:
base_sample.groupby("bank")["sentence_id"].nunique().rename("unique_sentence_ids")

bank
ecb    100
fed    100
Name: unique_sentence_ids, dtype: int64

In [23]:
duplicate_checks = []
for labeler in LABELERS:
    for bank in ["ecb", "fed"]:
        labels = read_labeler_labels(bank, labeler)
        duplicates = labels[labels.duplicated(subset=["sentence_id"], keep=False)].sort_values("sentence_id")
        duplicate_checks.append(
            {
                "labeler": labeler,
                "bank": bank,
                "rows": len(labels),
                "unique_sentence_ids": labels["sentence_id"].nunique(),
                "duplicate_sentence_id_rows": len(duplicates),
            }
        )

duplicate_summary = pd.DataFrame(duplicate_checks)
duplicate_summary

,labeler,bank,rows,unique_sentence_ids,duplicate_sentence_id_rows
0,daniel,ecb,100,100,0
1,daniel,fed,100,100,0
2,eric,ecb,100,100,0
3,eric,fed,100,100,0
4,sam,ecb,100,100,0
5,sam,fed,100,100,0


In [24]:
agreement_df = base_sample.drop(columns=["label"], errors="ignore").copy()
for labeler in LABELERS:
    label_frames = []
    for bank in ["ecb", "fed"]:
        label_frame = read_labeler_labels(bank, labeler).copy()
        label_frame["bank"] = bank
        label_frames.append(label_frame)
    label_df = pd.concat(label_frames, ignore_index=True)
    agreement_df = agreement_df.merge(label_df, on=["bank", "sentence_id"], how="left")

agreement_df.head()

,id,sentence_id,date,bank,chair,day_of_week,doc_word_count,sentence_word_count,text,daniel_label,eric_label,sam_label
0,20150305-ecb,20150305-ecb_105,2015-03-05,ecb,Mario Draghi,Thu,4662,9,As I said the ECB is a rules-based institution.,neutral,neutral,neutral
1,20150415-ecb,20150415-ecb_249,2015-04-15,ecb,Mario Draghi,Wed,5800,56,When I hear that the level of interest rates could be an incentive or a disincentive for countries governments to ch...,neutral,hawkish,hawkish
2,20150415-ecb,20150415-ecb_83,2015-04-15,ecb,Mario Draghi,Wed,5800,12,"Purchases are intended, intended, to run until the end of September 2016.",dovish,neutral,neutral
3,20150603-ecb,20150603-ecb_117,2015-06-03,ecb,Mario Draghi,Wed,4931,21,"But we should not forget that the structural component of our unemployment is high, and was high even before the cri...",neutral,hawkish,neutral
4,20150603-ecb,20150603-ecb_202,2015-06-03,ecb,Mario Draghi,Wed,4931,5,We are a rules-based institution.,neutral,neutral,neutral


In [25]:
label_cols = [f"{labeler}_label" for labeler in LABELERS]

missing_summary = agreement_df.groupby("bank")[label_cols].apply(lambda df: df.isna().sum())
missing_summary

,daniel_label,eric_label,sam_label
bank,,,
ecb,0,0,0
fed,0,0,0


In [26]:
complete_cases = agreement_df.dropna(subset=label_cols).copy()
complete_cases["all_three_match"] = complete_cases[label_cols].nunique(axis=1) == 1

print(f"Rows in base sample: {len(agreement_df)}")
print(f"Complete cases with all three labels: {len(complete_cases)}")
print(f"Overall exact agreement: {complete_cases['all_three_match'].mean():.3f}")
complete_cases.groupby("bank")["all_three_match"].mean().rename("exact_agreement")

Rows in base sample: 200
Complete cases with all three labels: 200
Overall exact agreement: 0.660


bank
ecb    0.61
fed    0.71
Name: exact_agreement, dtype: float64

In [27]:
pairwise_rows = []
for bank_name, bank_df in agreement_df.groupby("bank"):
    for left, right in combinations(LABELERS, 2):
        left_col = f"{left}_label"
        right_col = f"{right}_label"
        pair_df = bank_df.dropna(subset=[left_col, right_col])
        pairwise_rows.append(
            {
                "bank": bank_name,
                "labeler_a": left,
                "labeler_b": right,
                "n": len(pair_df),
                "percent_agreement": (pair_df[left_col] == pair_df[right_col]).mean(),
                "cohens_kappa": cohens_kappa(pair_df[left_col], pair_df[right_col]),
            }
        )

pairwise_by_bank = pd.DataFrame(pairwise_rows)
pairwise_by_bank

,bank,labeler_a,labeler_b,n,percent_agreement,cohens_kappa
0,ecb,daniel,eric,100,0.74,0.413886
1,ecb,daniel,sam,100,0.73,0.365005
2,ecb,eric,sam,100,0.70,0.328859
3,fed,daniel,eric,100,0.75,0.366126
4,fed,daniel,sam,100,0.84,0.541416
5,fed,eric,sam,100,0.79,0.397245


In [28]:
overall_pairwise = []
for left, right in combinations(LABELERS, 2):
    left_col = f"{left}_label"
    right_col = f"{right}_label"
    pair_df = agreement_df.dropna(subset=[left_col, right_col])
    overall_pairwise.append(
        {
            "labeler_a": left,
            "labeler_b": right,
            "n": len(pair_df),
            "percent_agreement": (pair_df[left_col] == pair_df[right_col]).mean(),
            "cohens_kappa": cohens_kappa(pair_df[left_col], pair_df[right_col]),
        }
    )

pd.DataFrame(overall_pairwise)

,labeler_a,labeler_b,n,percent_agreement,cohens_kappa
0,daniel,eric,200,0.745,0.391045
1,daniel,sam,200,0.785,0.445161
2,eric,sam,200,0.745,0.364090


In [29]:
missing_rows = agreement_df.loc[agreement_df[label_cols].isna().any(axis=1), ["bank", "sentence_id", *label_cols, "text"]]
missing_rows.head(20)

,bank,sentence_id,daniel_label,eric_label,sam_label,text


In [30]:
disagreements = complete_cases.loc[~complete_cases["all_three_match"], ["bank", "sentence_id", *label_cols, "text"]]
disagreements.head(20)

,bank,sentence_id,daniel_label,eric_label,sam_label,text
1,ecb,20150415-ecb_249,neutral,hawkish,hawkish,When I hear that the level of interest rates could be an incentive or a disincentive for countries governments to ch...
2,ecb,20150415-ecb_83,dovish,neutral,neutral,"Purchases are intended, intended, to run until the end of September 2016."
3,ecb,20150603-ecb_117,neutral,hawkish,neutral,"But we should not forget that the structural component of our unemployment is high, and was high even before the cri..."
7,ecb,20150716-ecb_69,neutral,neutral,hawkish,They didn't pay the IMF; what happens if Monday comes and goes and they don't pay you back?
9,ecb,20150903-ecb_223,neutral,hawkish,neutral,"One is through the trade channel, weakening the economies of the rest of the world, because China by now is a large ..."
10,ecb,20151022-ecb_50,neutral,neutral,dovish,Governor Nowotny last week said that monetary policy may be coming up to its limits and perhaps it was up to fiscal ...
13,ecb,20160121-ecb_133,neutral,neutral,dovish,"The markets seem to disagree completely with your assessment, as they keep selling bank shares."
15,ecb,20160421-ecb_145,dovish,neutral,dovish,"By the way, I would urge all the actors in this sector to resist the temptation to blame low interest rates as the c..."
17,ecb,20161020-ecb_183,neutral,dovish,neutral,"We certainly continue to monitor financial markets, and so far we haven't seen evidence of bubbles."
21,ecb,20170309-ecb_58,dovish,dovish,neutral,"By the way, incidentally, in just giving you these numbers about employment, let me add, those who had doubts about ..."


## 1. Label distribution by labeler × bank

Share of hawkish / dovish / neutral labels assigned by each labeler, broken out by ECB and Fed. High neutral rates across the board would suggest the corpus skews neutral and inflate raw agreement statistics.

In [31]:
dist_rows = []
for labeler in LABELERS:
    for bank in ["ecb", "fed"]:
        col = f"{labeler}_label"
        bank_df = complete_cases[complete_cases["bank"] == bank]
        counts = bank_df[col].value_counts(normalize=True)
        dist_rows.append({
            "labeler": labeler,
            "bank": bank,
            "hawkish": counts.get("hawkish", 0.0),
            "dovish": counts.get("dovish", 0.0),
            "neutral": counts.get("neutral", 0.0),
        })

dist_df = pd.DataFrame(dist_rows).set_index(["labeler", "bank"])
dist_df = dist_df[["hawkish", "dovish", "neutral"]]
dist_df.style.format("{:.1%}")

## 2. Agreement vs. all-neutral baseline

If the corpus is heavily neutral, two labelers can agree at a high rate even if both just stamp everything "neutral." The baseline below computes what pairwise percent-agreement would be if every labeler had assigned neutral to every sentence — that is, it equals the share of sentences where both labelers actually agreed on neutral, which in the all-neutral scenario is 100%. We compare actual percent-agreement against this naive baseline to see how much of our agreement is driven by neutral mass.

In [32]:
baseline_rows = []
for bank_name, bank_df in complete_cases.groupby("bank"):
    for left, right in combinations(LABELERS, 2):
        left_col = f"{left}_label"
        right_col = f"{right}_label"
        actual_agreement = (bank_df[left_col] == bank_df[right_col]).mean()
        # Both-neutral mass: fraction of sentences where both labelers independently said neutral.
        # In a world where everyone stamps everything neutral this would equal 1.0,
        # so the gap (actual - both_neutral) is the "signal" agreement above neutral mass.
        both_neutral = ((bank_df[left_col] == "neutral") & (bank_df[right_col] == "neutral")).mean()
        baseline_rows.append({
            "bank": bank_name,
            "labeler_a": left,
            "labeler_b": right,
            "n": len(bank_df),
            "actual_pct_agreement": actual_agreement,
            "both_neutral_rate": both_neutral,
            "lift_over_neutral_mass": actual_agreement - both_neutral,
        })

baseline_df = pd.DataFrame(baseline_rows)
baseline_df.style.format({
    "actual_pct_agreement": "{:.1%}",
    "both_neutral_rate": "{:.1%}",
    "lift_over_neutral_mass": "{:.1%}",
})

,bank,labeler_a,labeler_b,n,actual_pct_agreement,both_neutral_rate,lift_over_neutral_mass
0,ecb,daniel,eric,100,74.0%,62.0%,12.0%
1,ecb,daniel,sam,100,73.0%,60.0%,13.0%
2,ecb,eric,sam,100,70.0%,58.0%,12.0%
3,fed,daniel,eric,100,75.0%,66.0%,9.0%
4,fed,daniel,sam,100,84.0%,72.0%,12.0%
5,fed,eric,sam,100,79.0%,71.0%,8.0%


## 3. Agreement decomposed: salience then direction

Two separate questions:
- **3a. Salience** — do labelers agree on whether a sentence is opinionated at all? (neutral vs. non-neutral)
- **3b. Direction** — when both labelers think a sentence is opinionated, do they agree on hawkish vs. dovish?

In [33]:
# 3a: Salience agreement — binary neutral vs. non-neutral
salience_rows = []
for bank_name, bank_df in complete_cases.groupby("bank"):
    for left, right in combinations(LABELERS, 2):
        left_col = f"{left}_label"
        right_col = f"{right}_label"
        left_binary = (bank_df[left_col] != "neutral").map({True: "opinionated", False: "neutral"})
        right_binary = (bank_df[right_col] != "neutral").map({True: "opinionated", False: "neutral"})
        salience_rows.append({
            "bank": bank_name,
            "labeler_a": left,
            "labeler_b": right,
            "n": len(bank_df),
            "pct_agreement": (left_binary == right_binary).mean(),
            "cohens_kappa": cohens_kappa(left_binary, right_binary),
        })

salience_df = pd.DataFrame(salience_rows)
print("3a. Salience (neutral vs. non-neutral)")
salience_df.style.format({"pct_agreement": "{:.1%}", "cohens_kappa": "{:.3f}"})

3a. Salience (neutral vs. non-neutral)


,bank,labeler_a,labeler_b,n,pct_agreement,cohens_kappa
0,ecb,daniel,eric,100,80.0%,0.505
1,ecb,daniel,sam,100,73.0%,0.307
2,ecb,eric,sam,100,73.0%,0.338
3,fed,daniel,eric,100,80.0%,0.452
4,fed,daniel,sam,100,85.0%,0.543
5,fed,eric,sam,100,83.0%,0.482


In [34]:
# 3b: Direction agreement — hawkish vs. dovish, only where BOTH labelers are non-neutral
direction_rows = []
for bank_name, bank_df in complete_cases.groupby("bank"):
    for left, right in combinations(LABELERS, 2):
        left_col = f"{left}_label"
        right_col = f"{right}_label"
        both_nonneutral = bank_df[(bank_df[left_col] != "neutral") & (bank_df[right_col] != "neutral")]
        n = len(both_nonneutral)
        direction_rows.append({
            "bank": bank_name,
            "labeler_a": left,
            "labeler_b": right,
            "n_both_opinionated": n,
            "pct_agreement": (both_nonneutral[left_col] == both_nonneutral[right_col]).mean() if n > 0 else float("nan"),
            "cohens_kappa": cohens_kappa(both_nonneutral[left_col], both_nonneutral[right_col]),
        })

direction_df = pd.DataFrame(direction_rows)
print("3b. Direction (hawkish vs. dovish | both labelers non-neutral)")
direction_df.style.format({"pct_agreement": "{:.1%}", "cohens_kappa": "{:.3f}"})

3b. Direction (hawkish vs. dovish | both labelers non-neutral)


,bank,labeler_a,labeler_b,n_both_opinionated,pct_agreement,cohens_kappa
0,ecb,daniel,eric,18,66.7%,0.357
1,ecb,daniel,sam,13,100.0%,1.000
2,ecb,eric,sam,15,80.0%,0.587
3,fed,daniel,eric,14,64.3%,0.286
4,fed,daniel,sam,13,92.3%,0.843
5,fed,eric,sam,12,66.7%,0.351


## 4. Synthesis: Fleiss' kappa + per-labeler bias vs. majority vote

Fleiss' kappa gives a single multi-rater agreement score. The bias table shows how often each labeler agreed with the majority label, broken down by what the majority said — revealing systematic over- or under-assignment of a category relative to the group.

In [35]:
def fleiss_kappa(df: pd.DataFrame, label_cols: list[str], categories: list[str]) -> float:
    """Fleiss' kappa for fixed number of raters across all rows in df."""
    n_subjects = len(df)
    n_raters = len(label_cols)
    # Count how many raters assigned each category per row
    counts = pd.DataFrame(
        {cat: (df[label_cols] == cat).sum(axis=1) for cat in categories}
    )
    # Overall proportion each category was assigned
    p_j = counts.sum() / (n_subjects * n_raters)
    # Per-subject agreement
    P_i = ((counts ** 2).sum(axis=1) - n_raters) / (n_raters * (n_raters - 1))
    P_bar = P_i.mean()
    P_e = (p_j ** 2).sum()
    if P_e == 1:
        return float("nan")
    return (P_bar - P_e) / (1 - P_e)


CATEGORIES = ["hawkish", "dovish", "neutral"]

# Fleiss' kappa by bank and overall
fleiss_rows = []
for bank_name, bank_df in complete_cases.groupby("bank"):
    k = fleiss_kappa(bank_df, label_cols, CATEGORIES)
    fleiss_rows.append({"bank": bank_name, "n": len(bank_df), "fleiss_kappa": k})
k_overall = fleiss_kappa(complete_cases, label_cols, CATEGORIES)
fleiss_rows.append({"bank": "overall", "n": len(complete_cases), "fleiss_kappa": k_overall})

print("Fleiss' kappa (all three labelers simultaneously)")
pd.DataFrame(fleiss_rows).style.format({"fleiss_kappa": "{:.3f}"})

Fleiss' kappa (all three labelers simultaneously)


,bank,n,fleiss_kappa
0,ecb,100,0.368
1,fed,100,0.430
2,overall,200,0.399


In [ ]:
# Exclude 3-way ties — majority vote is arbitrary for those
has_majority = complete_cases[label_cols].nunique(axis=1) < 3
consensus_cases = complete_cases[has_majority].copy()
consensus_cases["majority_label"] = consensus_cases[label_cols].apply(
    lambda row: row.mode().iloc[0], axis=1
)

n_ties = (~has_majority).sum()
print(f"Excluded {n_ties} 3-way ties ({n_ties}/{len(complete_cases)} sentences have no consensus)")
print(f"Bias table covers {len(consensus_cases)} sentences with a genuine 2-of-3 majority\n")

# Per-labeler agreement with majority, broken down by majority label
bias_rows = []
for labeler in LABELERS:
    col = f"{labeler}_label"
    for majority_cat in CATEGORIES:
        subset = consensus_cases[consensus_cases["majority_label"] == majority_cat]
        n = len(subset)
        agreed = (subset[col] == majority_cat).sum()
        disagreed = subset[subset[col] != majority_cat][col].value_counts().to_dict()
        bias_rows.append({
            "labeler": labeler,
            "majority_label": majority_cat,
            "n_sentences": n,
            "agreed_with_majority": agreed,
            "agreement_rate": agreed / n if n > 0 else float("nan"),
            "said_instead": str(disagreed) if disagreed else "—",
        })

bias_df = pd.DataFrame(bias_rows)
print("Per-labeler agreement with majority vote, by majority label (excluding 3-way ties)")
bias_df.style.format({"agreement_rate": "{:.1%}"})

In [37]:
# Sentences where all three labelers disagreed (1-1-1 split) — majority vote is arbitrary for these
three_way_ties = complete_cases[complete_cases[label_cols].nunique(axis=1) == 3]
print(f"3-way ties (hawkish / dovish / neutral, one each): {len(three_way_ties)} / {len(complete_cases)}")
if len(three_way_ties) > 0:
    print()
    print(three_way_ties[["bank", "sentence_id", *label_cols, "majority_label", "text"]].to_string(index=False))

3-way ties (hawkish / dovish / neutral, one each): 9 / 200

bank      sentence_id daniel_label eric_label sam_label majority_label                                                                                                                                                                                                                                                                                                                                                                                                                            text
 ecb  20211216-ecb_26       dovish    hawkish   neutral         dovish                                                                                                                                                                                                                                                                                                           Net purchases under the PEPP could also be resumed, if necessary, to counter negativ

If you want a saved analysis table, you can export `agreement_df` with `agreement_df.to_csv("../output/labeling_agreement_joined.csv", index=False)` in a new cell.